# Wilcoxon: modelo combinado vs. baseline

Compara el modelo combinado contra el baseline con el metodo confirmado por la catedra: para cada semilla, el **maximo** valor observado en Kaggle entre los cortes probados (no un promedio, no un corte fijo compartido). Esos 10 maximos de cada modelo son los que se comparan de forma pareada con el test de Wilcoxon.

Los datos de `resultados_max_por_semilla.txt` son los scores reales observados en Kaggle para las 10 semillas de cada modelo (baseline: WF9500-WF9509, combinado: WF9820-WF9829), tomando de cada barrido de cortes el que dio el mayor score.

In [1]:
require("data.table")

tb <- fread("resultados_max_por_semilla.txt")
print(tb)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




    semilla corte_baseline ganancia_baseline corte_combinado ganancia_combinado
      <int>          <int>             <num>           <int>              <num>
 1:  187211           1900            84.153            1400             90.716
 2:  322247           2100            88.972            1500             95.867
 3:  346321           1500            87.642            2200             88.307
 4:  390263           1600            82.409            1500             98.027
 5:  430267           2100            93.042            1500             91.630
 6:  487649           1800            86.479            1900             92.211
 7:  522497           1700            89.387            1400             93.790
 8:  569321           2100            88.972            1400             91.215
 9:  906839           2100            88.722            1800             89.885
10:  992689           1400            87.144            1600             93.208
    diferencia
         <num>
 1:      6

## Test de Wilcoxon pareado

H0: no hay diferencia entre el combinado y el baseline.
H1: el combinado tiene mayor ganancia que el baseline (unilateral, `alternative="greater"`).

In [2]:
mejora <- sum(tb$diferencia > 0)
cat("El combinado mejora en", mejora, "de", nrow(tb), "semillas\n")
cat("Diferencia promedio:", round(mean(tb$diferencia), 3), "\n\n")

resultado <- wilcox.test(tb$ganancia_combinado, tb$ganancia_baseline,
  paired = TRUE, alternative = "greater")

print(resultado)

El combinado mejora en 9 de 10 semillas
Diferencia promedio: 4.793 


	Wilcoxon signed rank exact test

data:  tb$ganancia_combinado and tb$ganancia_baseline
V = 52, p-value = 0.004883
alternative hypothesis: true location shift is greater than 0



## Conclusion

Con p-value < 0.05, se rechaza H0: el modelo combinado tiene una ganancia significativamente mayor que el baseline. Esta es la evidencia estadistica que justifica haber elegido el combinado (FEhist lag3/delta3/ma3 + exclusion de meses de pandemia + training_pct 0.5) como modelo final, y sobre el que despues se armo el ensemble de las 10 semillas (`armar_ensemble.ipynb`).